<a href="https://colab.research.google.com/github/we-human-centric/CursoPythonDatos_2026/blob/main/Dia%2012%20-%202026-05-08%20-%20Correlaciones%2C%20regresiones%20e%20intro%20a%20scikit-learn/clase_12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Día 12 · Correlaciones, regresiones e intro a scikit-learn

Hoy es el día del umbral. Pasamos de "puedo describir los datos" a "puedo construir algo que predice". Es un cambio de paradigma.

Hasta ahora: análisis exploratorio. Hoy: modelado predictivo. Scikit-learn es la librería que hace esto accesible. Y todas sus funciones siguen el mismo patrón: `.fit()`, `.predict()`, `.score()`. Aprendés uno, aprendés a usar todos.

Empezamos con lo más simple: regresión lineal. Que en realidad es solo: ajustar una línea a los puntos.

## Setup

In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["figure.dpi"] = 100

print("Setup completo ✓")
print(f"scikit-learn {__import__('sklearn').__version__}")

---

## La pregunta

Titanic: sabemos que edad, clase y sexo se relacionan con la tarifa. Pero ¿puedo **predecir** cuánto pagó alguien solo con esos datos? ¿Cuán bien funciona?

In [ ]:
titanic = sns.load_dataset("titanic").dropna(subset=["age", "fare"])
print(f"Shape: {titanic.shape}")
print(f"\nPrimeras 3 filas:")
titanic[["age", "pclass", "sex", "fare"]].head(3)

---

## 1. Correlación de Pearson

Antes de modelar, veo cómo se relacionan las variables. Pearson mide relación **lineal**: rango [-1, +1]. Un valor de 0.8 significa: "cuando una sube, la otra sube bastante consistentemente".

In [ ]:
r_age_fare = titanic["age"].corr(titanic["fare"])
print(f"Pearson(edad, tarifa) = {r_age_fare:.3f}")

r, p = stats.pearsonr(titanic["age"], titanic["fare"])
print(f"Pearson r = {r:.3f}, p-value = {p:.4f}")
print(f"Significativa: {'sí' if p < 0.05 else 'no'}")

plt.figure(figsize=(10, 4))
plt.scatter(titanic["age"], titanic["fare"], alpha=0.5, s=30)
z = np.polyfit(titanic["age"], titanic["fare"], 1)
p_fit = np.poly1d(z)
plt.plot(titanic["age"].sort_values(), p_fit(titanic["age"].sort_values()), "r-", lw=2, label="Fit lineal")
plt.xlabel("Edad (años)")
plt.ylabel("Tarifa (£)")
plt.title(f"Edad vs Tarifa (r={r:.3f})")
plt.legend()
plt.tight_layout()

### Ejercicio 1

Crea una matriz de correlación de Pearson para `age`, `fare`, `pclass`, `survived`. Visualiza con `sns.heatmap()`.

In [ ]:
cols_numericas = ["age", "fare", "pclass", "survived"]
corr_matrix = titanic[cols_numericas].___()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", center=0, vmin=-1, vmax=1, square=True)
plt.title("Matriz de correlación — Titanic")
plt.tight_layout()

In [ ]:
assert corr_matrix.shape == (4, 4)
print("✅ Ejercicio 1 completado")

In [ ]:
# %% [solution]
# corr_matrix = titanic[cols_numericas].corr()

---

## 2. Regresión lineal · ajustar una línea

Regresión lineal busca ajustar `y = β₀ + β₁·x` minimizando el error cuadrático. Los coeficientes son interpretables: β₁ = "cada unidad más de x se asocia con β₁ unidades más de y".

In [ ]:
X = titanic[["age"]].values
y = titanic["fare"].values

modelo = LinearRegression()
modelo.fit(X, y)

intercept = modelo.intercept_
coef = modelo.coef_[0]

print(f"Modelo: tarifa = {intercept:.2f} + {coef:.3f} * edad")
print(f"Interpretación: cada año más de edad se asocia con £{coef:.2f} más de tarifa.")

edad_prueba = np.array([[25], [50]])
pred = modelo.predict(edad_prueba)
print(f"\nPredicción: edad=25 → tarifa={pred[0]:.2f}, edad=50 → tarifa={pred[1]:.2f}")

### Ejercicio 2

Entrena una regresión lineal múltiple para predecir tarifa usando edad, clase y sexo (encoding sexo como 0/1). Reporta R² en test.

In [ ]:
df_model = titanic.copy()
df_model["sex_int"] = (df_model["sex"] == "female").astype(int)

X = df_model[["age", "pclass", "sex_int"]]
y = df_model["fare"]

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

lr = LinearRegression()
lr.___(X_tr, y_tr)

pred = lr.___(X_te)
r2 = r2_score(y_te, pred)
mae = mean_absolute_error(y_te, pred)

print(f"R² en test = {r2:.3f}")
print(f"MAE = {mae:.2f} (error promedio en £)")
print(f"\nCoeficientes:")
for name, coef in zip(X.columns, lr.coef_):
    print(f"  {name}: {coef:+.3f}")
print(f"  intercepto: {lr.intercept_:+.3f}")

In [ ]:
assert r2 > 0, "R² debe ser positivo"
print("✅ Ejercicio 2 completado")

In [ ]:
# %% [solution]
# lr.fit(X_tr, y_tr)
# pred = lr.predict(X_te)

---

## 3. Regresión logística · clasificación

LinearRegression predice valores continuos. Si tu objetivo es categórico (sí/no), usas LogisticRegression. Predice probabilidades [0, 1], no valores arbitrarios.

In [ ]:
df_clf = titanic.dropna(subset=["survived"]).copy()
df_clf["sex_int"] = (df_clf["sex"] == "female").astype(int)

X = df_clf[["age", "pclass", "sex_int"]]
y = df_clf["survived"]

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

clf = LogisticRegression(max_iter=1000)
clf.fit(X_tr, y_tr)

pred_class = clf.predict(X_te)
accuracy = accuracy_score(y_te, pred_class)

print(f"Accuracy en test = {accuracy:.3f}")
print(f"\nReporte de clasificación:")
print(classification_report(y_te, pred_class, target_names=["Murió", "Sobrevivió"]))

pred_proba = clf.predict_proba(X_te[:3])
print(f"\nPrimeras 3 probabilidades predichas:")
print(pd.DataFrame(pred_proba, columns=["P(murió)", "P(sobrevivió)"]))

### Ejercicio 3

Entrena un LogisticRegression para predecir supervivencia usando edad, clase, sexo y tarifa. Reporta accuracy.

In [ ]:
X = df_clf[["age", "pclass", "sex_int", "fare"]]
y = df_clf["survived"]

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

clf2 = LogisticRegression(max_iter=1000)
clf2.___(X_tr, y_tr)

acc = accuracy_score(y_te, clf2.___(X_te))
print(f"Accuracy = {acc:.3f}")

coefs = pd.Series(abs(clf2.coef_[0]), index=X.columns).sort_values(ascending=False)
print(f"\nFeatures más importantes:")
print(coefs)

In [ ]:
assert acc > 0
print("✅ Ejercicio 3 completado")

In [ ]:
# %% [solution]
# clf2.fit(X_tr, y_tr)
# acc = accuracy_score(y_te, clf2.predict(X_te))

---

## Desafío · Pipeline completo

Construye un **pipeline end-to-end**:

1. Prepara datos: limpieza, encoding categóricas, selección de features.
2. Estandariza (StandardScaler — importante para interpretabilidad).
3. Divide train/test con random_state=42.
4. Entrena regresión o clasificación.
5. Reporta métrica + top 3 features importantes.
6. Escribe conclusión en 3-4 oraciones.

Criterios:
- Random state 42 en todo.
- Features claros, sin duplicados.
- Métrica apropiada (R² o accuracy).
- Interpretación de coeficientes.

In [ ]:
# === PASO 1: Preparación ===
# Dataset: Titanic, tips, diamonds, penguins (o el tuyo)
# ← Tu código aquí

# === PASO 2: Feature engineering ===
# Selecciona 4-6 features numéricas o codifica categóricas
# ← Tu código aquí

# === PASO 3: Train/test split ===
# ← Tu código aquí

# === PASO 4: Estandarización ===
# ← Tu código aquí

# === PASO 5: Entrenamiento ===
# ← Tu código aquí

# === PASO 6: Evaluación ===
# ← Tu código aquí

# === PASO 7: Conclusión ===
# Escribe 3-4 oraciones aquí

print("\n🎯 Desafío: Entregado")

---

## Lo que aprendiste

Pearson mide relación lineal. LinearRegression para predecir números. LogisticRegression para clasificación. Train/test split con random_state=42 (siempre). StandardScaler para que los coeficientes sean comparables. Scikit-learn: mismo patrón en todo.

Mañana: ¿cómo sé que mi modelo es bueno? Métricas. Comparación. Validación cruzada.

---

## Trabajo en equipo

Para vuestro proyecto:

- Matriz de correlación de vuestro dataset.
- 3 features más prometedoras para predecir el objetivo.
- Modelo baseline (regresión o clasificación) con esos features.

Entrega: `baseline_model.ipynb` con datos, modelo, métrica y top 3 features.